# Question-to-Cypher (Q2C) — Execution Runner

Notebook untuk menghasilkan (*generate*) Cypher query dari input pertanyaan (`TEST_ID`, `QUESTION`, `CATEGORY`) dan langsung mengeksekusinya ke database Neo4j untuk melihat hasilnya secara langsung.

**Alur Kerja Runner:**
1. Load dataset test case (berisi `TEST_ID`, `QUESTION`, `CATEGORY`).
2. Generate Cypher query untuk setiap `QUESTION` secara langsung menggunakan model LLM dengan prompt dari Google Sheets.
3. Eksekusi Cypher query yang dihasilkan langsung ke Neo4j untuk mendapatkan **Actual Query Result** berupa list of dict (JSON).
4. Simpan laporan hasil eksekusi ke CSV dan Google Sheets.

In [1]:
# === Setup ===
import sys
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from neo4j import GraphDatabase

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'Project root: {PROJECT_ROOT}')

Project root: d:\TA\llm-driven-legal-kg-visualization


## Step 0: Configuration

Set `EXPERIMENT_ID` dan sumber test data.

In [2]:
# === Configuration ===
EXPERIMENT_ID = "008"              # Unique ID per eksperimen
PROMPT_ID = "PROMPT_11"             # Prompt version from QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE
SCHEMA_ID = "KG_SCHEMA_3"          # Schema ID from KG_SCHEMA worksheet
DOCUMENT_IDS = ["POJK_11_2022", "UU_11_2008", "UU_19_2016"]

DELAY_BETWEEN_REQUESTS = 1                  # Delay antar request (seconds)

# Sumber test data: 'csv' atau 'gsheets'
TEST_DATA_SOURCE = "gsheets"                    # 'csv' atau 'gsheets'
CSV_PATH = "UU_11_2008_E2E_DATATEST_V3.csv"         # Path file test data Q2C (misal POJK_Q2C_TEST_DATA.csv)
GSHEETS_SHEET_NAME = "UU_11_2008_E2E_DATATEST_V3"   # Jika source = gsheets

# Tulis hasil ke Google Sheets?
WRITE_TO_GSHEETS = False
EXPERIMENT_SHEET_NAME = f"EXP_Q2C_RUN_{EXPERIMENT_ID}"

print(f'Experiment: {EXPERIMENT_ID}')
print(f'Document IDs: {DOCUMENT_IDS}')
print(f'Test data source: {TEST_DATA_SOURCE}')
print(f'Prompt ID: {PROMPT_ID}')


Experiment: 008
Document IDs: ['POJK_11_2022', 'UU_11_2008', 'UU_19_2016']
Test data source: gsheets
Prompt ID: PROMPT_11


## Step 1: Load Test Data

In [3]:
# === Load Test Data ===
if TEST_DATA_SOURCE == "gsheets":
    from modules.google_sheets_utils import GoogleUtil
    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
    test_df = gu.load_dataframe_from_sheet(spreadsheet_id, GSHEETS_SHEET_NAME)
else:
    # Look in project root, parent directory, or data directory
    if os.path.exists(CSV_PATH):
        test_df = pd.read_csv(CSV_PATH)
    elif os.path.exists(PROJECT_ROOT / CSV_PATH):
        test_df = pd.read_csv(PROJECT_ROOT / CSV_PATH)
    else:
        test_df = pd.read_csv(PROJECT_ROOT.parent / CSV_PATH)

# Pastikan kolom dasar yang diperlukan ada
required_cols = ['TEST_ID', 'QUESTION', 'CATEGORY']
for col in required_cols:
    if col not in test_df.columns:
        raise ValueError(f'Kolom wajib {col} tidak ditemukan dalam data.')

print(f'Loaded {len(test_df)} test cases')
print(f'Columns: {list(test_df.columns)}')
test_df[required_cols].head(5)

2026-06-01 11:13:38,893 - INFO - Retrieving worksheet 'UU_11_2008_E2E_DATATEST_V3' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-06-01 11:13:40,987 - INFO - Successfully loaded 178 rows from worksheet 'UU_11_2008_E2E_DATATEST_V3'.


Loaded 178 test cases
Columns: ['TEST_ID', 'CATEGORY', 'QUESTION']


,TEST_ID,QUESTION,CATEGORY
0,UU_11_2008_E2E_V3_001,"Mau nanya dong, uu ite itu sebenarnya mengatur...",Regulasi
1,UU_11_2008_E2E_V3_002,Apa saja ketentuan yang diatur di dalam bab i ...,BAB I KETENTUAN UMUM
2,UU_11_2008_E2E_V3_003,Tolong rangkum isi dari bab II uu 11/2008 dala...,BAB II ASAS DAN TUJUAN
3,UU_11_2008_E2E_V3_004,"Mau nanya dong, apa saja yang dibahas di bab t...","BAB III INFORMASI, DOKUMEN, DAN TANDA TANGAN E..."
4,UU_11_2008_E2E_V3_005,Bisa jelaskan secara ringkas isi bab 4 uu no 1...,BAB IV PENYELENGGARAAN SERTIFIKASI ELEKTRONIK ...


## Step 2: Load Prompt Template from Google Sheets

Mengambil system prompt dan user prompt template yang terdaftar di Google Sheets.

In [4]:
# === Load Prompt from GSheets ===
from modules.prompt_fetcher import fetch_question_to_cypher_prompt
from pipeline.transform.prompt_builder import load_schema_from_gsheets
from modules.google_sheets_utils import GoogleUtil

prompt_data = fetch_question_to_cypher_prompt(PROMPT_ID)

SYSTEM_PROMPT = prompt_data['SYSTEM_PROMPT']
USER_PROMPT_TEMPLATE = prompt_data.get('USER_PROMPT', 'Question: {question}')

# Fetch KG Schema from Google Sheets and format System Prompt
if '{KG_SCHEMA}' in SYSTEM_PROMPT:
    try:
        gu_schema = GoogleUtil(
            private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
            client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
        )
        spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
        kg_schema_str = load_schema_from_gsheets(SCHEMA_ID, gu_schema, spreadsheet_id)
        SYSTEM_PROMPT = SYSTEM_PROMPT.replace('{KG_SCHEMA}', kg_schema_str)
        print("✅ Formatted {KG_SCHEMA} in SYSTEM_PROMPT using Google Sheets (KG_SCHEMA_2)")
    except Exception as e:
        print(f"⚠️ Failed to load KG_SCHEMA from Google Sheets: {e}. Fallback to local config.")
        from pipeline.transform.prompt_builder import load_schema_from_file
        kg_schema_str = load_schema_from_file(PROJECT_ROOT / 'config' / 'kg_schema.json')
        SYSTEM_PROMPT = SYSTEM_PROMPT.replace('{KG_SCHEMA}', kg_schema_str)
        print("✅ Fallback: Formatted {KG_SCHEMA} using config/kg_schema.json")

print(f'System prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'User prompt template length: {len(USER_PROMPT_TEMPLATE)} chars')


2026-06-01 11:13:41,063 - INFO - Fetching prompt 'PROMPT_11' from sheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE'...
2026-06-01 11:13:41,064 - INFO - Retrieving worksheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-06-01 11:13:42,741 - INFO - Successfully loaded 11 rows from worksheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE'.
2026-06-01 11:13:42,745 - INFO - Loaded prompt 'PROMPT_11': ['PROMPT_ID', 'SYSTEM_PROMPT', 'USER_PROMPT', 'NOTES']
2026-06-01 11:13:44,421 - INFO - Loaded schema 'KG_SCHEMA_3' (1358 chars)


✅ Formatted {KG_SCHEMA} in SYSTEM_PROMPT using Google Sheets (KG_SCHEMA_2)
System prompt length: 21530 chars
User prompt template length: 125 chars


## Step 3: Connect to Neo4j

In [5]:
# === Connect to Neo4j ===
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run('RETURN 1 AS test')
    print(f'✅ Connected to Neo4j database: "{NEO4J_DATABASE}"')

✅ Connected to Neo4j database: "experiment-2"


## Step 4: Define Helper Functions

In [6]:
# === Cypher Gen & Execution Helpers ===
import re
import google.generativeai as genai

# Setup Gemini model
genai.configure(api_key=os.getenv('GEMINI_API_KEY', ''))
model = genai.GenerativeModel('gemini-2.5-flash')
print('✅ LLM model ready: gemini-2.5-flash')

def clean_cypher(text: str) -> str:
    """Strip markdown code blocks and extra whitespace from LLM output.
    Matches backend LLMService._clean_cypher() exactly.
    """
    # Remove ```cypher ... ``` or ```sql ... ``` or ``` ... ```
    pattern = r'```(?:cypher|sql|plaintext)?\s*\n?(.*?)```'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Fallback: if starts with ``` but no closing, strip first line
    if text.startswith('```'):
        lines = text.split('\n')
        return '\n'.join(lines[1:]).strip().rstrip('`')
    return text.strip()

def is_valid_cypher(query: str) -> bool:
    """Basic validation: has RETURN, balanced parens/brackets/braces, not empty, and no invalid trailing syntax.
    Matches backend LLMService._is_valid_cypher() exactly.
    """
    if not query or 'RETURN' not in query.upper():
        return False
    if query.count('(') != query.count(')'):
        return False
    if query.count('[') != query.count(']'):
        return False
    if query.count('{') != query.count('}'):
        return False

    # Check for truncated or invalid trailing syntax
    clean_query = query.strip()
    if clean_query.endswith('.') or clean_query.endswith(',') or clean_query.endswith('+') or clean_query.endswith('-'):
        return False

    # Check for trailing logical operators or incomplete clauses
    upper_query = clean_query.upper()
    invalid_trailing_words = [' AND', ' OR', ' WHERE', ' MATCH', ' WITH', ' RETURN']
    for word in invalid_trailing_words:
        if upper_query.endswith(word):
            return False

    return True

def generate_cypher_offline(question: str, doc_ids: list, max_retries: int = 2) -> str:
    """Generate Cypher query from question using direct LLM call.
    Matches backend LLMService.generate_cypher() logic:
    - Uses GSheet prompt template for system prompt
    - Same max_output_tokens (4096) as backend
    - Same retry logic: if first attempt is invalid, retry with simpler prompt
    - Rate-limit handling with exponential backoff
    """
    # Construct doc filter instruction if applicable
    doc_filter_instruction = ''
    if doc_ids:
        ids_str = ', '.join(f"'{d}'" for d in doc_ids)
        doc_filter_instruction = (
            f'\n\nCRITICAL: The user has selected specific document sources. '
            f'You MUST add this filter to your first MATCH clause: '
            f'WHERE <node>.source_document_id IN [{ids_str}]\n'
            f'Apply this to the main entity node (Pasal, Ayat, Bab, or Regulasi) in the query.'
        )

    # Format user prompt from GSheet template
    user_prompt = USER_PROMPT_TEMPLATE
    if '{question}' in user_prompt:
        user_prompt = user_prompt.replace('{question}', question)
    else:
        user_prompt += f'\nQuestion: {question}'

    if '{doc_filter_instruction}' in user_prompt:
        user_prompt = user_prompt.replace('{doc_filter_instruction}', doc_filter_instruction)
    elif doc_filter_instruction:
        user_prompt += doc_filter_instruction

    for attempt in range(max_retries):
        try:
            # Attempt 1: Use full prompt template
            response = model.generate_content(
                [SYSTEM_PROMPT, user_prompt],
                generation_config={'temperature': 0.0, 'max_output_tokens': 4096},
            )
            cypher = clean_cypher(response.text.strip())

            if is_valid_cypher(cypher):
                return cypher

            # Invalid Cypher — retry with simpler prompt (matches backend retry logic)
            print(f'  [attempt {attempt+1}: invalid Cypher, retrying with simpler prompt]')
            retry_prompt = (
                f'Question: {question}\n\n'
                f'Write a SIMPLE Cypher query (1-3 lines) for Neo4j. '
                f'Write MATCH ... RETURN ... LIMIT 100 directly. NO markdown, NO explanation, NO thinking.'
            )
            response = model.generate_content(
                [SYSTEM_PROMPT, retry_prompt],
                generation_config={'temperature': 0.0, 'max_output_tokens': 4096},
            )
            cypher = clean_cypher(response.text.strip())
            if is_valid_cypher(cypher):
                return cypher

        except Exception as e:
            if 'quota' in str(e).lower() or '429' in str(e):
                wait = 5 * (attempt + 1)
                print(f'  [rate limit, waiting {wait}s]', end='')
                time.sleep(wait)
            else:
                raise
    return ''

def run_cypher_query(driver, query: str, database: str) -> list:
    """Execute Cypher query and return results as list of dicts."""
    with driver.session(database=database) as session:
        result = session.run(query)
        return [dict(record) for record in result]


✅ LLM model ready: gemini-2.5-flash


c:\Users\daffarafi\miniconda3\envs\ta-skripsi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\daffarafi\AppData\Local\Temp\ipykernel_34436\2642513030.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Step 5: Run Cypher Generation and Execution

In [7]:
# === Run Generation & Execution Loop ===
results = []
total_cases = len(test_df)

print(f"Starting Q2C Execution Runner on {total_cases} test cases...\n")

for i, row in test_df.iterrows():
    test_id = row['TEST_ID']
    question = row['QUESTION']
    category = row['CATEGORY']
    
    print(f"[{i+1}/{total_cases}] Running {test_id} ({category})...")
    
    # Step 1: Generate Cypher query directly using Google Sheets prompt template
    try:
        generated_cypher = generate_cypher_offline(question, DOCUMENT_IDS)
        if not generated_cypher:
            raise ValueError("Empty or invalid Cypher generated")
    except Exception as e:
        generated_cypher = ""
        print(f"  ❌ Generation Error: {e}")
        results.append({
            "TEST_ID": test_id,
            "QUESTION": question,
            "CATEGORY": category,
            "GENERATED_CYPHER": "",
            "EXEC_SUCCESS": False,
            "ERROR": str(e),
            "ACTUAL_RESULT": "[]",
            "FORMATTED_ACTUAL_QUERY_RESULT": "[]"
        })
        continue
        
    print(f"  Generated Cypher: {generated_cypher}")
    
    # Step 2: Execute query in Neo4j
    exec_success = False
    actual_results = []
    error_msg = ""
    
    try:
        actual_results = run_cypher_query(driver, generated_cypher, NEO4J_DATABASE)
        exec_success = True
        print(f"  Execution: Success ({len(actual_results)} rows)")
    except Exception as e:
        error_msg = str(e)
        print(f"  ❌ Execution Error: {error_msg[:100]}...")
        
    results.append({
        "TEST_ID": test_id,
        "QUESTION": question,
        "CATEGORY": category,
        "GENERATED_CYPHER": generated_cypher,
        "EXEC_SUCCESS": exec_success,
        "ERROR": error_msg,
        "ACTUAL_RESULT": json.dumps(actual_results, ensure_ascii=False),
        "FORMATTED_ACTUAL_QUERY_RESULT": json.dumps(actual_results, indent=2, ensure_ascii=False)
    })
    
    # Delay to avoid overloading
    time.sleep(DELAY_BETWEEN_REQUESTS)

results_df = pd.DataFrame(results)
print("\n=== Runner Completed! ===")

Starting Q2C Execution Runner on 178 test cases...

[1/178] Running UU_11_2008_E2E_V3_001 (Regulasi)...
  Generated Cypher: MATCH (r:Regulasi) WHERE r.source_document_id = 'UU_11_2008'
OPTIONAL MATCH (r)-[:MEMUAT]->(b:Bab)
RETURN r.label AS regulasi, r.content AS detail, b.label AS daftar_bab, r.source_document_id AS regulasi_id
ORDER BY b.label
LIMIT 100
  Execution: Success (13 rows)
[2/178] Running UU_11_2008_E2E_V3_002 (BAB I KETENTUAN UMUM)...
  Generated Cypher: MATCH (b:Bab)-[:MEMUAT*1..2]->(p:Pasal) WHERE b.source_document_id = 'UU_11_2008' AND b.label =~ '(?i)^BAB I(\\s.*|$)' OPTIONAL MATCH (p)-[:MEMILIKI_AYAT]->(a:Ayat) OPTIONAL MATCH (p)-[:MENGATUR]->(ph_p:PerbuatanHukum) OPTIONAL MATCH (a)-[:MENGATUR]->(ph_a:PerbuatanHukum) RETURN b.label AS bab, p.label AS pasal, a.label AS ayat, COLLECT(DISTINCT ph_p.label) AS perbuatan_diatur_pasal, COLLECT(DISTINCT ph_a.label) AS perbuatan_diatur_ayat, b.source_document_id AS regulasi ORDER BY p.label, a.label LIMIT 100
  Execution: Suc

KeyboardInterrupt: 

## Step 6: Summary and Samples

In [ ]:
# === Overall Summary ===
total = len(results_df)
exec_success = len(results_df[results_df['EXEC_SUCCESS'] == True])
exec_failed = total - exec_success

print(f'╔══════════════════════════════════════════════════════════════╗')
print(f'║  Q2C Execution Runner Report                                 ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  Experiment: {EXPERIMENT_ID:<47s} ║')
print(f'║  Database:   {NEO4J_DATABASE:<47s} ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  Total Cases:       {total:>3d}                                      ║')
print(f'║  Execution Success:  {exec_success:>3d}  ({exec_success/total:.1%})                             ║')
print(f'║  Execution Failed:   {exec_failed:>3d}  ({exec_failed/total:.1%})                             ║')
print(f'╚══════════════════════════════════════════════════════════════╝')

In [ ]:
# === Print Execution Samples ===
print("Sample of first 10 execution results:\n")
for _, row in results_df.head(10).iterrows():
    print(f"Test ID: {row['TEST_ID']} ({row['CATEGORY']})")
    print(f"Question: {row['QUESTION']}")
    print(f"Generated Cypher: {row['GENERATED_CYPHER']}")
    print(f"Execution Success: {row['EXEC_SUCCESS']}")
    if row['ERROR']:
        print(f"Error: {row['ERROR']}")
    else:
        # Display pretty JSON
        res_obj = json.loads(row['ACTUAL_RESULT'])
        print(f"Result: {json.dumps(res_obj[:3], indent=2, ensure_ascii=False)} (showing up to 3 rows)")
    print("-" * 80)

## Step 7: Save Laporan Hasil Eksekusi

In [ ]:
# === Save Results ===
output_dir = 'data/evaluation'
os.makedirs(output_dir, exist_ok=True)

output_csv = f'{output_dir}/q2c_exec_run_{EXPERIMENT_ID}.csv'
results_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f'✅ Results saved to: {output_csv}')

if WRITE_TO_GSHEETS:
    try:
        from modules.google_sheets_utils import GoogleUtil, GoogleSheetsWriter
        gu = GoogleUtil(
            private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
            client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
        )
        spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
        writer = GoogleSheetsWriter(
            google_util=gu,
            sheet_id=spreadsheet_id,
            worksheet_name=EXPERIMENT_SHEET_NAME,
            batch_size=5,
        )
        writer.write_dataframe(results_df)
        print(f'✅ Results uploaded to Google Sheets: {EXPERIMENT_SHEET_NAME}')
    except Exception as e:
        print(f"⚠️ Failed to upload to Google Sheets: {e}")

In [ ]:
# === Cleanup ===
driver.close()
print("✅ Neo4j connection driver closed.")